# Agentic RAG, OpenAI Agents SDK, MCP

My 3 favorite things in 12 minutes.

Please see the README for setup instructions.

### Video series:

1. Comparing Agentic RAG with traditional RAG:  
https://youtu.be/K6wpRkJrcpM

2. THIS VIDEO: Building the initial version of this:  
https://youtu.be/KdKqMfs-8gs

3. Taking Agentic RAG to the next level with more tools.   
https://youtu.be/UmGLirGbKTk  

In [1]:
import notebook_setup  # Windows Jupyter MCP fix — must run before agents import

from agents import Agent, Runner, SQLiteSession
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import os
import sys
from IPython.display import display, Markdown
from pathlib import Path
from qdrant_client import QdrantClient
from agents.extensions.models.litellm_model import LitellmModel
import gradio as gr

load_dotenv(override=True)

True

## Picking your LLM provider, including Cerebras

If you'd like to use Cerebras, the high speed inference provider, with open-source model gpt-oss-120b, then sign up for a Cerebras account here:  
https://cloud.cerebras.ai/

Alternatively, to use OpenAI models like gpt-5.4-mini, replace the entire contents of the next cell with:

```python
model = "gpt-5.4-mini"
```

You can also use OpenRouter:

```python
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
model = LitellmModel(model="openrouter/openrouter-model-name", api_key=openrouter_api_key)
```


In [2]:
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
model = LitellmModel(model="cerebras/gpt-oss-120b", api_key=cerebras_api_key)

In [3]:
agent = Agent("Tester", model=model)
response = await Runner.run(agent, "what is 2+2?")
print(response.final_output)

2 + 2 = 4.


## Which URLs to examine for content for our Agent?

Here I specify some of my websites. You should provide whichever have the information that you want in your agent's memory. I put the same URL in the list multiple times so that the agent stores more memories.

In [4]:
urls = ["https://edwarddonner.com", "https://edwarddonner.com/about"] * 3 + ["https://edwarddonner.com/curriculum"] * 10
urls

['https://edwarddonner.com',
 'https://edwarddonner.com/about',
 'https://edwarddonner.com',
 'https://edwarddonner.com/about',
 'https://edwarddonner.com',
 'https://edwarddonner.com/about',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum']

## The MCP Parameters

In [5]:
knowledge_dir = Path.cwd() / "knowledge"
knowledge_dir.mkdir(exist_ok=True)
vectordb_path = knowledge_dir / "vectordb"

mcp_bin = Path(sys.executable).parent
fetch_exe = mcp_bin / ("mcp-server-fetch.exe" if sys.platform == "win32" else "mcp-server-fetch")
qdrant_exe = mcp_bin / ("mcp-server-qdrant.exe" if sys.platform == "win32" else "mcp-server-qdrant")

fetch_params = {
    "command": str(fetch_exe),
    "args": [],
}

vectorstore_params = {
    "command": str(qdrant_exe),
    "args": [],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

In [6]:
CONTEXT = """
You are an Agent with expert knowledge about Ed Donner with particular focus on his online AI courses.
"""

INSTRUCTIONS = CONTEXT + """
You are populating your memories with information retrieved from a given website.
Use your MCP tools to retrieve the website. Extract key knowledge. Check what's already in your memories to avoid duplicates.
After you are done, reply with a brief status update and the number of memories you added.
Aim to add at least 10 unique memories, unless your existing memories are already comprehensive.
"""

In [7]:
async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=120) as fetch_mcp:
    tools = await fetch_mcp.list_tools()
tools

[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get 

In [8]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
    tools = await vectorstore_mcp.list_tools()
tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'additionalProperties': False, 'properties': {'query': {'description': 'What to search for', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema={'properties': {'result': {'items': {'type': 'string'}, 'type': 'array'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'additionalProperties': False, 'properties': {'information': {'description': 'Text to store', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'defa

## The Ingest Agent

In [9]:
for url in urls:
    async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=120) as fetch_mcp:
        async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
            agent = Agent(name="Ingester", model=model, instructions=INSTRUCTIONS, mcp_servers=[fetch_mcp, vectorstore_mcp])
            task = f"Add unique memories with information from this website: {url} and reply with a one sentence status update including how many memories were added."
            response = await Runner.run(agent, task, max_turns=50)
            display(Markdown(response.final_output))

Added 11 unique memories from edwarddonner.com.

Added 10 unique memories.

Added 10 new unique memories.

Added 10 new unique memories from the site; knowledge base now includes key details about Edward Donner’s roles, ventures, speaking work, and mission.

Added 10 unique memories.

CancelledError: 

In [10]:
client = QdrantClient(path=str(vectordb_path))
collection_name = "knowledge"

info = client.get_collection(collection_name)
print(f"Memories in '{collection_name}': {info.points_count}\n")

points, _ = client.scroll(collection_name=collection_name, limit=200, with_payload=True, with_vectors=False)
for i, p in enumerate(points, 1):
    doc = (p.payload or {}).get("document", "")
    preview = doc.replace("\n", " ")[:160]
    print(f"{i:>3}. {preview}{'...' if len(doc) > 160 else ''}")

client.close()

Memories in 'knowledge': 57

  1. Untapt was selected for the Accenture FinTech Innovation Lab accelerator program and was recognized as an American Banker Top 20 Company To Watch.
  2. Ed Donner posted resources on May 28, 2025 titled 'Which order to take the AI courses?'.
  3. Nebula.io uses a patented AI model that matches people with roles without relying on keywords, offering greater accuracy and speed.
  4. Ed Donner created Udemy AI courses that have enrolled approximately 900,000 students.
  5. Edward Donner founded the AI startup untapt in 2013, building talent marketplaces and data science software for recruitment firms, initially focusing on tech ro...
  6. Ed Donner served as a Managing Director at JPMorgan before his AI startup ventures.
  7. Nebula.io’s mission is driven by the fact that 77% of people do not feel inspired or engaged at work, aiming to align individuals with their Ikigai.
  8. Ed Donner’s long‑term goal is to help people discover their purpose (Ikigai) and

In [11]:
EXPERT_INSTRUCTIONS = """
You are an expert about Ed Donner and his online courses. You are answering questions about him and his courses to visitors on his website.
Use your memories to help answer the question. If you don't know the answer, say so.
"""

EXAMPLES = ["Which course covers RAG?", "Which course covers MCP?", "How do I become an expert on Claude Code?"]

In [12]:
convo = SQLiteSession("test_conversation")

## Agentic RAG + OpenAI Agents SDK + MCP in a 4 line function!

In [13]:
async def chat(message, history):
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
        agent = Agent(name="Expert", model=model, instructions=EXPERT_INSTRUCTIONS, mcp_servers=[vectorstore_mcp])
        response = await Runner.run(agent, message, session=convo)
        return response.final_output

In [14]:
from styles import CSS, JS
gr.ChatInterface(chat, examples=EXAMPLES, chatbot=gr.Chatbot(show_label=False, height=700)).launch(css=CSS, js=JS, theme=gr.themes.Base(), inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Video series:

1. Comparing Agentic RAG with traditional RAG:  
https://youtu.be/K6wpRkJrcpM

2. THIS VIDEO: Building the initial version of this:  
https://youtu.be/KdKqMfs-8gs

3. Taking Agentic RAG to the next level with more tools.   
https://youtu.be/UmGLirGbKTk  